# Qwen QLoRA training

Load the local Qwen3-4B-Instruct-2507, prepare the existing **60 train / 10 val / 30 test** splits, and train one completion-only QLoRA adapter.

Select this project's **.venv** kernel in VS Code and run from the TheologyDistillment folder. The demonstration defaults to **8 training examples and 10 optimizer steps**. Set TRAIN_EXAMPLE_LIMIT to None to use all 60 training examples.

The **Run training** cell starts optimization. Validation and test prompts remain separate and unused by Trainer. There are no evaluations or generated answers in this notebook. For another run, restart the kernel and choose a new RUN_NAME.


In [ ]:
import hashlib
import json
from dataclasses import dataclass
from importlib.metadata import version
from pathlib import Path

import torch
from datasets import DatasetDict, load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    PreTrainedTokenizerBase,
    Trainer,
    TrainingArguments,
    set_seed,
)


In [ ]:
ROOT = Path.cwd()
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_DIR = ROOT / "models" / "Qwen3-4B-Instruct-2507"
DATA_FILES = {
    "train": ROOT / "data/train/nicene.draft.jsonl",
    "val": ROOT / "data/val/prompts.draft.jsonl",
    "test": ROOT / "data/test/prompts.draft.jsonl",
}
RUN_NAME = "nicene-notebook-smoke"
RUN_DIR = ROOT / "models" / RUN_NAME

SEED = 42
TRAIN_EXAMPLE_LIMIT = 8  # None selects all 60 training examples.
MAX_STEPS = 10

set_seed(SEED)


## 1. Load Qwen

NF4 stores the base model's linear weights in 4 bits; bfloat16 is the computation dtype. LoRA will add the trainable parameters later. This configuration uses one CUDA GPU and loads only local model files.

The tokenizer's chat template supplies Qwen's role markers.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
tokenizer.padding_side = "right"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    quantization_config=quantization_config,
    dtype=torch.bfloat16,
    device_map={"": 0},
    attn_implementation="sdpa",
)
model.config.use_cache = False


## 2. Load the datasets

Each file becomes a Hugging Face **Dataset**; a **DatasetDict** holds the named splits. These classes already provide indexing and preprocessing, so no custom dataset wrapper is needed.

Keep the existing split assignments. Training records contain reference answers; val and test contain prompts only. The eight-example demonstration subset is selected within train.


In [ ]:
datasets = DatasetDict({
    split: load_dataset(
        "json",
        data_files=str(path),
        split="train",
        cache_dir=str(ROOT / "models" / ".dataset_cache"),
    )
    for split, path in DATA_FILES.items()
})

training_source = datasets["train"]
if TRAIN_EXAMPLE_LIMIT is not None:
    training_source = training_source.select(range(TRAIN_EXAMPLE_LIMIT))

datasets


## 3. Prepare completion-only labels

The full conversation is the model input. Prompt labels are set to **-100**, so the loss supervises only the assistant answer and its ending token. Qwen shifts the targets internally for next-token prediction.

The small preview below displays the text that receives supervision.


In [ ]:
def encode_completion(example):
    messages = example["messages"]
    prompt_ids = tokenizer.apply_chat_template(
        messages[:-1], tokenize=True, add_generation_prompt=True, return_dict=False,
    )
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=False, return_dict=False,
    )

    # Keep the assistant's final EOS, dropping the template's trailing newline.
    completion_end = len(input_ids) - input_ids[::-1].index(tokenizer.eos_token_id)
    input_ids = input_ids[:completion_end]

    return {
        "input_ids": input_ids,
        "attention_mask": [1] * len(input_ids),
        "labels": [-100] * len(prompt_ids) + input_ids[len(prompt_ids):],
    }


train_dataset = training_source.map(
    encode_completion,
    remove_columns=training_source.column_names,
    keep_in_memory=True,
)

supervised_ids = [token for token in train_dataset[0]["labels"] if token != -100]
print(tokenizer.decode(supervised_ids))


## 4. Batch the examples

The collator pads inputs to the longest example in each batch and pads labels with **-100**. It preserves the completion mask created above.


In [ ]:
@dataclass
class CompletionOnlyCollator:
    tokenizer: PreTrainedTokenizerBase

    def __call__(self, examples):
        batch = self.tokenizer.pad(
            [
                {"input_ids": row["input_ids"], "attention_mask": row["attention_mask"]}
                for row in examples
            ],
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt",
        )
        batch_length = batch["input_ids"].shape[1]
        batch["labels"] = torch.tensor(
            [
                row["labels"] + [-100] * (batch_length - len(row["labels"]))
                for row in examples
            ],
            dtype=torch.long,
        )
        return batch


## 5. Configure QLoRA and Trainer

Freeze the base weights and attach a rank-16 LoRA adapter. Gradient checkpointing reduces activation memory by recomputing some operations during backpropagation.

Trainer shuffles the training examples, computes completion-token cross-entropy, and updates the adapter using AdamW. Four batches of one example are accumulated per optimizer step. The constant learning rate and ten-step default match the existing smoke configuration.

Calling train() on the model enables training behavior such as dropout; Trainer's train() method performs the optimization.


In [ ]:
model = prepare_model_for_kbit_training(
    model, gradient_checkpointing_kwargs={"use_reentrant": False},
)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.train()
model.print_trainable_parameters()


In [ ]:
training_args = TrainingArguments(
    output_dir=str(RUN_DIR),
    run_name=RUN_NAME,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="constant",
    optim="adamw_torch",
    max_grad_norm=1.0,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=0,
    logging_steps=1,
    save_strategy="no",
    eval_strategy="no",
    report_to="none",
)

# Create a fresh output folder; an existing run will not be overwritten.
RUN_DIR.mkdir(parents=True, exist_ok=False)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=CompletionOnlyCollator(tokenizer),
    processing_class=tokenizer,
)


## 6. Run training

Executing this cell starts the configured training run. The reported loss is training loss; it does not measure doctrinal accuracy, genuine belief, or moral improvement.


In [ ]:
training_result = trainer.train()


## 7. Save the adapter

Save the adapter, tokenizer, training settings, and a compact record of the inputs and environment. The base model is not copied. Training metrics include measured runtime; no evaluation metrics are produced.


In [ ]:
adapter_dir = RUN_DIR / "adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
trainer.save_metrics("train", training_result.metrics)
(RUN_DIR / "training_args.json").write_text(training_args.to_json_string(), encoding="utf-8")

revision_file = MODEL_DIR / ".cache/huggingface/download/config.json.metadata"
run_info = {
    "model_id": MODEL_ID,
    "config_source_revision": (
        revision_file.read_text(encoding="utf-8").splitlines()[0]
        if revision_file.exists() else None
    ),
    "quantization": quantization_config.to_dict(),
    "training_ids": list(training_source["id"]),
    "data_sha256": {
        split: hashlib.sha256(path.read_bytes()).hexdigest()
        for split, path in DATA_FILES.items()
    },
    "chat_template_sha256": hashlib.sha256(tokenizer.chat_template.encode()).hexdigest(),
    "versions": {name: version(name) for name in (
        "torch", "transformers", "datasets", "peft", "bitsandbytes", "accelerate"
    )},
    "gpu": torch.cuda.get_device_name(0),
}
(RUN_DIR / "run_info.json").write_text(json.dumps(run_info, indent=2), encoding="utf-8")
print("Saved adapter:", adapter_dir)
